# Solafune Tree Canopy Project



In [6]:
# Directories Setup within '/content/'
# Import Packages
from pathlib import Path
import shutil
import os
from datetime import datetime
import numpy as np
import time
import yaml
import torch
import pandas as pd
import ultralytics
from ultralytics import YOLO
import json
from PIL import Image

REPO_ROOT = Path("../")                              # default for Colab; will be overridden by Papermill if needed
REPO_DIR = ("../")                                    # Non-Path Repo Directory

from ultralytics.utils import SETTINGS  # <-- works on current releases
SETTINGS.update({"datasets_dir": REPO_DIR})

SETTINGS


{'settings_version': '0.0.6',
 'datasets_dir': '../',
 'weights_dir': 'C:\\Users\\hmanasi1\\Documents\\ADML\\Project\\weights',
 'runs_dir': 'C:\\Users\\hmanasi1\\Documents\\ADML\\Project\\runs',
 'uuid': 'f76199480971710b55317b68bedd0bfd095b200aa13bd5223b77f5b287e3a41c',
 'sync': True,
 'api_key': '',
 'openai_api_key': '',
 'clearml': True,
 'comet': True,
 'dvc': True,
 'hub': True,
 'mlflow': True,
 'neptune': True,
 'raytune': True,
 'tensorboard': False,
 'wandb': False,
 'vscode_msg': True,
 'openvino_msg': True}

In [7]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device name:", torch.cuda.get_device_name(0))
    print("Device count:", torch.cuda.device_count())
    print("Current device:", torch.cuda.current_device())
else:
    print("Using CPU")


CUDA available: True
Device name: NVIDIA RTX A2000 12GB
Device count: 1
Current device: 0


In [8]:
!nvidia-smi

Mon Nov  3 01:56:44 2025       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 516.01       Driver Version: 516.01       CUDA Version: 11.7     |
|-------------------------------+----------------------+----------------------+
| GPU  Name            TCC/WDDM | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA RTX A200... WDDM  | 00000000:B3:00.0 Off |                  Off |
| 30%   30C    P8     6W /  70W |   1653MiB / 12282MiB |      8%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

## 02_Train_Model

### Model Training Parameters

Open the `train_model_overrides` file to modify for training parameters to your desire

/solafune_tree_canopy/configurations/train_model_overrides.yaml

### Training Model Py File

Training Model is set to `YOLO11s-seg`. Open the training model to modify desired YOLO model on `Row 21`

/solafune_tree_canopy/scripts/02_train_model.py

### Train Model

In [9]:
  #-- Configurations --#
local_configs_path = Path("../configurations/")

  #-- Output Runs --#
local_runs_output = Path("../runs/")

In [10]:
# Run Training Model File
train_model = f"{REPO_ROOT}/scripts/02_train_model.py"

import ultralytics
from ultralytics import YOLO
import numpy as np
import time
import yaml
from pathlib import Path

DATA_CONFIG_PATH = REPO_ROOT / 'configurations' / 'model_data-seg.yaml'

# Load model parameters / overrides
with open(REPO_ROOT / 'configurations' / 'train_model_overrides.yaml', 'r') as f:
    overrides = yaml.safe_load(f)

# override the path in your dictionary before calling train
overrides['data'] = str(DATA_CONFIG_PATH)

# Load a pretrained model
model = YOLO('yolo11m-seg.pt') # yolo version 11s segementation

# Train the model
results = model.train(**overrides)

# State output location
try:
    print("Saved to:", model.trainer.save_dir)
except Exception:
    pass


New https://pypi.org/project/ultralytics/8.3.223 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.185  Python-3.11.14 torch-2.0.1+cu117 CUDA:0 (NVIDIA RTX A2000 12GB, 12281MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=..\configurations\model_data-seg.yaml, degrees=180, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.028, hsv_s=0.9, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.003, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m-seg.pt, momentum=0.937, mosaic=0.9, multi_scale=False, name=train_Yolo11m_canopy_adamW

train: Scanning C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\data\processed\labels\train.cache... 120 images, 0 backgrounds, 0 corrupt: 100%|██████████| 120/120 [00:00<?, ?it/s


val: Fast image access  (ping: 0.10.0 ms, read: 1214.8114.5 MB/s, size: 3072.3 KB)


val: Scanning C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\data\processed\labels\val.cache... 30 images, 0 backgrounds, 0 corrupt: 100%|██████████| 30/30 [00:00<?, ?it/s]


optimizer: 'optimizer=auto' found, ignoring 'lr0=0.003' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.001667, momentum=0.9) with parameter groups 115 weight(decay=0.0), 126 weight(decay=0.000515625), 125 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to runs\segment\train_Yolo11m_canopy_adamW_
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss  Instances       Size


...

                   all         30       7960       0.46      0.308      0.333       0.17      0.433      0.286      0.309      0.146



100 epochs completed in 0.459 hours.
Optimizer stripped from runs\segment\train_Yolo11m_canopy_adamW_\weights\last.pt, 45.2MB
Optimizer stripped from runs\segment\train_Yolo11m_canopy_adamW_\weights\best.pt, 45.2MB

Validating runs\segment\train_Yolo11m_canopy_adamW_\weights\best.pt...
Ultralytics 8.3.185  Python-3.11.14 torch-2.0.1+cu117 CUDA:0 (NVIDIA RTX A2000 12GB, 12281MiB)
YOLO11m-seg summary (fused): 138 layers, 22,336,854 parameters, 0 gradients, 112.9 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:01<00:00,  2.80it/s]

                   all         30       7960      0.461      0.308      0.333      0.171      0.436      0.286      0.309      0.146
       individual_tree         30       7104      0.555      0.397       0.45      0.237      0.495      0.351      0.399      0.195
        group_of_trees         26        856      0.366      0.218      0.215      0.104      0.376      0.221      0.219     0.0967
Speed: 0.3ms preprocess, 17.5ms inference, 0.0ms loss, 7.4ms postprocess per image


Saved to: runs\segment\train_Yolo11m_canopy_adamW_
